# Phase 3: Sentence Embeddings with SBERT

**Goal:** Generate high-quality single vectors for entire sentences — the core ingredient of RAG.

**What you'll learn:**
- Why vanilla BERT is not ideal for sentence similarity (it wasn't trained for that)
- How SBERT (Sentence-BERT) fixes this with siamese network training
- How to embed sentences and compute semantic similarity
- How to find the most semantically similar sentence to a query
- Visualizing sentence clusters in 2D

**This phase is the direct prerequisite to Phase 4 (RAG).**

## Step 1: Install & Import

In [ ]:
# Install (run once)
# !pip install sentence-transformers matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer, util

print('All imports successful!')

## Step 2: Load SBERT Model

`all-MiniLM-L6-v2` is a great default:
- Only 22MB — very fast
- Produces 384-dimensional sentence vectors
- Trained specifically to make semantically similar sentences have similar vectors

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

# Test it
test_embedding = model.encode("Hello world")
print(f'Model loaded!')
print(f'Embedding dimension: {test_embedding.shape[0]}')
print(f'Type: {type(test_embedding)}')

## Step 3: Basic Sentence Similarity

Let's start with the most fundamental operation: how similar are two sentences?

In [ ]:
sentence_pairs = [
    # High similarity pairs
    ("A dog is playing in the park", "A puppy is running outside"),
    ("The stock market fell sharply", "Equity markets declined significantly"),
    ("BERT is a transformer model", "BERT uses self-attention to understand language"),

    # Low similarity pairs
    ("The cat sat on the mat", "The stock market crashed today"),
    ("I love pizza", "Neural networks need training data"),

    # Tricky pairs (same words, different meaning)
    ("The bank was steep", "The bank approved my loan"),
]

print(f'{"Pair":<5} {"Similarity":>10}  Sentences')
print('=' * 80)

for i, (s1, s2) in enumerate(sentence_pairs):
    e1 = model.encode(s1)
    e2 = model.encode(s2)
    sim = util.cos_sim(e1, e2).item()
    label = '🟢 similar' if sim > 0.5 else '🔴 different'
    print(f'  {i+1}   {sim:>8.4f}  {label}')
    print(f'        A: "{s1}"')
    print(f'        B: "{s2}"')
    print()

## Step 4: Encode a Corpus and Find Top-K Similar Sentences

This is the exact operation that a RAG retriever performs:
embed a query, find the closest chunks from a pre-embedded corpus.

In [ ]:
# This simulates our 'knowledge base'
corpus = [
    # AI/ML
    "BERT is a transformer model pre-trained on masked language modeling.",
    "GPT models generate text by predicting the next token.",
    "Neural networks learn by adjusting weights using backpropagation.",
    "Transformers use self-attention to relate each word to every other word.",
    "Fine-tuning adapts a pre-trained model to a specific task with less data.",
    "Word2Vec trains word embeddings using surrounding context words.",
    "RAG combines a retriever and a generator to answer questions from documents.",
    "Cosine similarity measures the angle between two embedding vectors.",

    # Finance
    "The Federal Reserve sets interest rates to control inflation.",
    "A bull market refers to a period of rising stock prices.",
    "Diversification reduces investment risk by spreading across asset classes.",

    # Health
    "Regular exercise reduces the risk of cardiovascular disease.",
    "A balanced diet includes proteins, carbohydrates, fats, and vitamins.",
    "Sleep is critical for memory consolidation and brain health.",
]

# Encode the full corpus at once (batch encoding is faster)
print('Encoding corpus...')
corpus_embeddings = model.encode(corpus, show_progress_bar=True)
print(f'Corpus embeddings shape: {corpus_embeddings.shape}')
print(f'  = {corpus_embeddings.shape[0]} sentences × {corpus_embeddings.shape[1]} dimensions')

In [ ]:
def semantic_search(query, corpus, corpus_embeddings, top_k=3):
    """Find the top-k most similar sentences to the query."""
    # Embed the query
    query_embedding = model.encode(query)

    # Compute cosine similarity between query and all corpus embeddings
    similarities = util.cos_sim(query_embedding, corpus_embeddings)[0]

    # Get top-k results sorted by similarity
    top_results = similarities.topk(top_k)

    return [(corpus[idx], similarities[idx].item()) for idx in top_results.indices]


# Run several queries
queries = [
    "How do language models understand context?",
    "What is the best way to invest money?",
    "How does retrieval augmented generation work?",
]

for query in queries:
    print(f'Query: "{query}"')
    results = semantic_search(query, corpus, corpus_embeddings, top_k=3)
    for rank, (sentence, score) in enumerate(results, 1):
        print(f'  #{rank} (score={score:.4f}): {sentence}')
    print()

## Step 5: Visualize Sentence Clusters with PCA

In [ ]:
# Color-code by topic
colors_map = (['royalblue'] * 8) + (['forestgreen'] * 3) + (['crimson'] * 3)
labels_map = (['AI/ML'] * 8) + (['Finance'] * 3) + (['Health'] * 3)

# Reduce 384-dim vectors to 2D
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(corpus_embeddings)

fig, ax = plt.subplots(figsize=(13, 8))

# Plot each group
for group, color in [('AI/ML', 'royalblue'), ('Finance', 'forestgreen'), ('Health', 'crimson')]:
    idxs = [i for i, l in enumerate(labels_map) if l == group]
    ax.scatter(coords_2d[idxs, 0], coords_2d[idxs, 1],
               c=color, label=group, s=120, alpha=0.8, zorder=5)

# Annotate with shortened text
for i, sentence in enumerate(corpus):
    short = sentence[:45] + '...' if len(sentence) > 45 else sentence
    ax.annotate(short, (coords_2d[i, 0], coords_2d[i, 1]),
                textcoords='offset points', xytext=(5, 4), fontsize=7.5, color='#333333')

ax.set_title('SBERT Sentence Embeddings — PCA Visualization\n(sentences of similar topics should cluster together)', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('phase3_sentence_clusters.png', dpi=150)
plt.show()

## Step 6: Build the Similarity Matrix

In [ ]:
# Pick a subset for cleaner visualization
subset_idx = [0, 2, 3, 6, 7, 8, 10, 11, 13]  # pick diverse sentences
subset_sentences = [corpus[i] for i in subset_idx]
subset_embeddings = corpus_embeddings[subset_idx]

# Compute pairwise cosine similarity matrix
sim_matrix = util.cos_sim(subset_embeddings, subset_embeddings).numpy()

# Plot heatmap
short_labels = [s[:40] + '...' for s in subset_sentences]

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0.0, vmax=1.0)
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)

for i in range(len(short_labels)):
    for j in range(len(short_labels)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if sim_matrix[i,j] > 0.6 else 'black')

ax.set_title('Pairwise Sentence Similarity Matrix\n(1.0 = identical, 0.0 = completely different)', fontsize=12)
plt.tight_layout()
plt.savefig('phase3_similarity_matrix.png', dpi=150)
plt.show()

## Step 7: Save Corpus Embeddings to Disk

In a real RAG system, you pre-compute and save embeddings so you don't redo this work every time. We'll save them as numpy files — Phase 4 will load these directly.

In [ ]:
# Save embeddings and corpus text for reuse in Phase 4
np.save('corpus_embeddings.npy', corpus_embeddings)

import json
with open('corpus_sentences.json', 'w') as f:
    json.dump(corpus, f, indent=2)

print('Saved:')
print('  corpus_embeddings.npy  — numpy array of shape', corpus_embeddings.shape)
print('  corpus_sentences.json  — list of', len(corpus), 'sentences')
print()
print('Phase 4 will load these files to build the vector store!')

## Why SBERT? — BERT vs SBERT

### The Problem with Raw BERT for Sentence Similarity

BERT was never trained to produce geometrically meaningful sentence vectors. To compare two sentences with raw BERT, you have to feed them **together** in one forward pass:

```
Input:  [CLS] Sentence A [SEP] Sentence B [SEP]
Output: classification logit (are they related?)
```

This means comparing a query against 10,000 sentences requires **10,000 separate BERT forward passes** — one per pair. At ~65ms each, that is 10 minutes per query. Completely impractical for search.

Even with mean pooling, raw BERT sentence vectors are geometrically poor — two sentences with opposite meanings can score 0.9 cosine similarity because BERT was never trained to care about the distance between sentence vectors.

---

### How SBERT Fixes This — The Siamese Network

SBERT fine-tunes BERT using a **siamese network**: the same BERT model processes each sentence independently, and a contrastive loss function trains the weights so that similar sentences land near each other in vector space.

```
Sentence A ──► BERT ──► mean pool ──► vector A ──┐
                                                   ├──► cosine similarity ──► loss
Sentence B ──► BERT ──► mean pool ──► vector B ──┘
```

The loss pushes similar pairs together and dissimilar pairs apart. After fine-tuning, you can encode every sentence **once**, store the vectors, and compare a new query using a single dot product across the whole index.

```
Encode corpus (once):   N sentences  ──► N vectors  (stored on disk)

At query time:          query  ──► 1 vector  ──► dot product against all N  ──► top-k
```

---

### Practical Differences

| | Raw BERT | SBERT |
|---|---|---|
| Sentence comparison | Feed both together, 1 pass | Encode independently, compare vectors |
| Search over 10,000 sentences | 10,000 forward passes (~10 min) | 1 forward pass + dot products (~ms) |
| Raw sentence vector quality | Poor — not trained for it | Good — explicitly trained for it |
| Typical use case | Classification, QA, NER | Semantic search, clustering, RAG |
| Embedding dimension | 768 (bert-base) | 384 (all-MiniLM-L6-v2) |
| Phase in this project | Phase 2 | Phase 3 and 4 |

> **Key insight:** SBERT does not change the transformer architecture. It changes what the model was trained to optimise — making the geometry of the embedding space meaningful so that "the cat sat on the mat" and "a kitten rested on the rug" end up near each other.


## Key Takeaways

1. **SBERT produces fixed-size sentence vectors** — regardless of sentence length, you get a single 384-dim vector
2. **Semantically similar sentences are close** — cosine similarity works directly as a relevance score
3. **Batch encoding is efficient** — encode your whole corpus once, then just embed queries at search time
4. **Clusters emerge naturally** — sentences from the same topic group together without any supervision
5. **Saved embeddings = your vector index** — this is exactly what RAG systems store in their vector databases

**→ Phase 4 builds the full RAG system using these embeddings!**